# Compare Biospecimen Results: New Additions (Mar 2026 vs Jun 2023)

This notebook compares two biospecimen analysis CSVs and lists **new additions**: rows that appear in the Mar 2026 file but not in the Jun 2023 file. Rows are compared using a composite key (PATNO, CLINICAL_EVENT, TYPE, TESTNAME, TESTVALUE, RUNDATE).

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
FILE_2026 = DATA_DIR / "Current_Biospecimen_Analysis_Results_06Mar2026.csv"
FILE_2023 = DATA_DIR / "Current_Biospecimen_Analysis_Results_29Jun2023.csv"

In [14]:
df_2026 = pd.read_csv(FILE_2026, quoting=1, low_memory=False)
df_2023 = pd.read_csv(FILE_2023, quoting=1, low_memory=False)

print("Mar 2026 shape:", df_2026.shape)
print("Jun 2023 shape:", df_2023.shape)
print("Columns:", list(df_2026.columns))

Mar 2026 shape: (1010327, 13)
Jun 2023 shape: (409182, 13)
Columns: ['PATNO', 'SEX', 'COHORT', 'CLINICAL_EVENT', 'TYPE', 'TESTNAME', 'TESTVALUE', 'UNITS', 'RUNDATE', 'PROJECTID', 'PI_NAME', 'PI_INSTITUTION', 'update_stamp']


In [3]:
KEY_COLS = ["PATNO", "CLINICAL_EVENT", "TYPE", "TESTNAME", "TESTVALUE", "RUNDATE"]

def make_row_key(row):
    return tuple(row[c] for c in KEY_COLS)

df_2026["row_key"] = df_2026.apply(make_row_key, axis=1)
df_2023["row_key"] = df_2023.apply(make_row_key, axis=1)

In [8]:
keys_2023 = set(df_2023["row_key"])
mask_new = ~df_2026["row_key"].isin(keys_2023)
new_only = df_2026.loc[mask_new].drop(columns=["row_key"]).copy()

print(f"New additions (in 2026 only): {len(new_only):,} rows")
print(f"Unique PATNO in new additions: {new_only['PATNO'].nunique():,}")
print(f"Unique PROJECTID in new additions: {new_only['PROJECTID'].nunique():,}")

New additions (in 2026 only): 731,149 rows
Unique PATNO in new additions: 2,779
Unique PROJECTID in new additions: 48


In [23]:
print("\n--- New PROJECTIDs ---")
new_only['PROJECTID'].unique()



--- New PROJECTIDs ---


array([105, 114, 118, 119, 124, 125, 134, 135, 144, 145, 148, 149, 159,
       161, 182, 195, 199, 202, 204, 208, 211, 213, 215, 221, 236, 240,
       243, 245, 248, 256, 259, 262, 264, 274, 276, 300, 301, 303, 283,
       302, 286, 127, 129, 235, 239, 307, 292, 104])

In [11]:
print("\n--- Breakdown by PROJECTID (top 20) ---")
print(new_only["PROJECTID"].value_counts().head(20).to_string())

print("\n--- Breakdown by TESTNAME (top 20) ---")
print(new_only["TESTNAME"].value_counts().head(20).to_string())

print("\n--- Breakdown by TYPE ---")
print(new_only["TYPE"].value_counts().to_string())

print("\n--- Breakdown by CLINICAL_EVENT ---")
print(new_only["CLINICAL_EVENT"].value_counts().to_string())

print("\n--- Breakdown by COHORT ---")
print(new_only["COHORT"].value_counts().to_string())


--- Breakdown by PROJECTID (top 20) ---
PROJECTID
256    466107
213     91680
135     65536
283     20522
119     15169
264      9360
105      7116
262      4920
276      4688
118      4459
292      4167
240      4064
114      4000
307      3820
245      3738
239      2761
302      2418
221      2230
204      1764
259      1734

--- Breakdown by TESTNAME (top 20) ---
TESTNAME
Ptau217p          4688
NFL               4144
GFAP              4144
ABeta42           4078
ABeta40           4078
pTau181           4078
pS65 Ubiquitin    4064
eMTBR-TAU243      2333
C16 Cer           2050
C16 GL2           2050
C16 GlcCer        2050
C16 SM            2050
C18 Cer           2050
C18 GL2           2050
C18 GlcCer        2050
C18 SM            2050
C20 Cer           2050
C20 GL2           2050
C20 GlcCer        2050
C20 SM            2050

--- Breakdown by TYPE ---
TYPE
Plasma                    612677
Cerebrospinal Fluid        63548
miRNA                      26285
Cell Line                   6

In [52]:
KEY_COLS = ["TYPE", "TESTNAME", "PROJECTID"]

def make_row_key(row):
    return tuple(row[c] for c in KEY_COLS)

df_2026["row_key_2"] = df_2026.apply(make_row_key, axis=1)
df_2023["row_key_2"] = df_2023.apply(make_row_key, axis=1)

In [91]:
test_count_df = df_2026["row_key_2"].value_counts().reset_index()
test_count_df[['TYPE', 'TESTNAME', 'PROJECTID']] = pd.DataFrame(test_count_df['row_key_2'].tolist(), index=test_count_df_full.index)
test_count_df = test_count_df[['TYPE', 'TESTNAME', 'PROJECTID','count']]
print(test_count_df.head())

test_count_df.to_csv('../output/ppmi_biospec_counts_per_test.csv', index=False)

                  TYPE  TESTNAME  PROJECTID  count
0                Serum       NfL        144   4795
1               Plasma  Ptau217p        276   4629
2               Plasma       NFL        283   4094
3               Plasma      GFAP        283   4094
4  Cerebrospinal Fluid   ABeta42        283   4034


In [130]:
summary_1 = df_2026["row_key_2"].value_counts()
summary_2 = df_2026.groupby("row_key_2").agg({'PATNO':'nunique','CLINICAL_EVENT':'nunique'})

summary = pd.merge(summary_1, summary_2, left_index=True, right_index=True)
summary.rename(columns={'count':'n_entries','PATNO':'n_subjects','CLINICAL_EVENT':'n_timepoints'}, inplace=True)

summary.reset_index(inplace=True)
row_ids = pd.DataFrame(summary['row_key_2'].tolist(), columns=['TYPE', 'TESTNAME', 'PROJECTID'])
summary = row_ids.merge(summary, how='left', left_index=True, right_index=True).drop(columns=['row_key_2'])

display(summary)

summary.to_csv('../output/ppmi_biospec_test_summaries.csv', index=False)

,TYPE,TESTNAME,PROJECTID,n_entries,n_subjects,n_timepoints
0,Serum,NfL,144,4795,1141,8
1,Plasma,Ptau217p,276,4629,1302,12
2,Plasma,NFL,283,4094,1581,8
3,Plasma,GFAP,283,4094,1581,8
4,Cerebrospinal Fluid,ABeta42,283,4034,1914,5
...,...,...,...,...,...,...
5107,Cell Line,Luminex VEGF MLi2 (rep2),161,1,1,1
5108,Cell Line,Pluritest,161,1,1,1
5109,Tissue Formalin,Diagnosis Based on Algorithm,211,1,1,1
5110,Tissue Formalin,Replicates Amplifying before 50h,211,1,1,1


## PPMI Ongoing Specimen Analysis table (from web)

The table on the [PPMI Ongoing Specimen Analysis](https://www.ppmi-info.org/access-data-specimens/ongoing-analysis/specimen-analysis) page is embedded in an iframe (Smartsheet). We load it from the iframe URL and keep the main data table.

In [37]:
pd.set_option('display.width', 1000)

In [42]:
# Table is in an iframe; this is the Smartsheet publish URL
PPMI_ONGOING_ANALYSIS_URL = "https://publish.smartsheet.com/0fbd4b92b37e46c0b747d74ede6b487f"

tables = pd.read_html(PPMI_ONGOING_ANALYSIS_URL)
# First table is the main specimen analysis table (291 rows); others are small UI elements
df_ongoing = tables[0].copy()

# Optional: drop the unnamed index column if you don't need it
if df_ongoing.columns[0] == "Unnamed: 0":
    df_ongoing = df_ongoing.drop(columns=["Unnamed: 0"])

print("Shape:", df_ongoing.shape)
print("Columns:", list(df_ongoing.columns))
df_ongoing.head(10)

Shape: (291, 11)
Columns: ['Principal Investigator', 'Organization', 'Project Title', 'Data Availability', 'Analyte', 'Matrix', 'Included Enrollment Cohorts', 'Subcategories included', 'Visit(s)', 'Project ID', 'Notes']


,Principal Investigator,Organization,Project Title,Data Availability,Analyte,Matrix,Included Enrollment Cohorts,Subcategories included,Visit(s),Project ID,Notes
0,"Andrew Singleton, PhD",NIA,APOE4 Genotyping​,Currently available,APOE4 genotyping​,DNA from whole blood,Healthy Control Parkinson Disease SWEDD (legacy),NaN,Baseline,104,NaN
1,NaN,APOE4 genotyping​,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Clemens Scherzer, MD",Harvard,Evaluation of PD-linked Transcripts in PPMI,Currently available,aSyn transcripts,RNA from whole blood,Healthy Control Parkinson Disease,NaN,Baseline,105,NaN
3,NaN,aSyn transcripts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Andrew Singleton, PhD",NIA,Selected Genetic Variants Genotyped Using Neur...,Currently available,NeuroX genotyping,DNA from whole blood,Healthy Control Parkinson Disease Prodromal SW...,Genetic subjects,Baseline,"106, 107",NaN
5,NaN,NeuroX genotyping,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Andrew Singleton, PhD",NIA,Immunochip Genotyping on DNA Samples from PPMI,Currently available,Immunochip genotyping​,DNA from whole blood,Healthy Control Parkinson Disease SWEDD (legacy),NaN,Baseline,108,NaN
7,NaN,Immunochip genotyping​,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"Les Shaw, PhD",UPenn,"A-beta 1-42, t-tau and p-tau 181 Measurements ...",Currently available,"Aβ, tau, & p-tau181 ​",CSF,Healthy Control Parkinson Disease Prodromal SW...,Genetic subjects,"Baseline, 6, 12, 24, 36 months","110, 125","Innogentics: PD, HC, & SWEDD through yr 1; Ele..."
9,NaN,"Aβ, tau, & p-tau181 ​",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
new_projects = new_only['PROJECTID'].unique()

# Convert new_projects to strings for matching
new_projects_str = set(str(pid) for pid in new_projects if pd.notnull(pid))

# Find rows where 'Project ID' string contains any of the new_projects entries
def contains_any_project_id(cell):
    if not pd.isna(cell):
        for pid in new_projects_str:
            if pid in str(cell):
                return True
    return False

mask = df_ongoing['Project ID'].apply(contains_any_project_id)
df_matching = df_ongoing[mask]

print("Rows in df_ongoing with Project ID containing new project ids (first 5):")
display(df_matching.head(5))

df_matching.to_csv('../output/ppmi_biospec_new_projects.csv', index=False)

Rows in df_ongoing with Project ID containing new project ids (first 5):


,Principal Investigator,Organization,Project Title,Data Availability,Analyte,Matrix,Included Enrollment Cohorts,Subcategories included,Visit(s),Project ID,Notes
0,"Andrew Singleton, PhD",NIA,APOE4 Genotyping​,Currently available,APOE4 genotyping​,DNA from whole blood,Healthy Control Parkinson Disease SWEDD (legacy),NaN,Baseline,104,NaN
2,"Clemens Scherzer, MD",Harvard,Evaluation of PD-linked Transcripts in PPMI,Currently available,aSyn transcripts,RNA from whole blood,Healthy Control Parkinson Disease,NaN,Baseline,105,NaN
8,"Les Shaw, PhD",UPenn,"A-beta 1-42, t-tau and p-tau 181 Measurements ...",Currently available,"Aβ, tau, & p-tau181 ​",CSF,Healthy Control Parkinson Disease Prodromal SW...,Genetic subjects,"Baseline, 6, 12, 24, 36 months","110, 125","Innogentics: PD, HC, & SWEDD through yr 1; Ele..."
14,"Judith Potashkin, PhD",Rosalind & Franklin University,Whole blood RNA Biomarkers of PD,Currently available,mRNA transcripts,RNA from whole blood,Healthy Control Parkinson Disease,NaN,Baseline,114,NaN
20,"Andrew Singleton, PhD",NIA,Whole Genome Sequencing & Analysis of PPMI Pro...,Currently available,Whole genome sequencing,DNA from whole blood,Healthy Control Parkinson Disease Prodromal SW...,Genetic subjects,Baseline,118,NaN
